# LSTM WTI Forecasting — Visualization Suite
Run each cell independently. Assumes `lstm_wti_h1.pkl`, `lstm_wti_h5.pkl`, `lstm_wti_h20.pkl` exist from running `lstm_wti_multihorizon.py`.

In [ ]:
# ============================================================
# CELL 1: Imports & Setup
# ============================================================
import numpy as np
import pandas as pd
import pickle
import warnings
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from scipy import stats
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linestyle': '--',
    'font.family': 'serif',
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
    'figure.dpi': 150,
})

COLORS = {
    'actual': '#1a1a2e', 'pred': '#e63946', 'fill': '#a8dadc',
    'pos': '#2a9d8f', 'neg': '#e76f51',
    'h1': '#264653', 'h5': '#2a9d8f', 'h20': '#e9c46a',
    'bar1': '#264653', 'bar2': '#e76f51',
}

MACRO_FILE = Path('MD_Test/2025-12-MD.csv')
PRICE_FILE = Path('Price/Price.csv')
FEATURES = [
    'RPI','W875RX1','CMRMTSPLx','IPFPNSS','USWTRADE','USTRADE','BUSLOANS',
    'CONSPI','S&P 500','S&P PE ratio','FEDFUNDS','TB3MS','TB6MS','GS1','GS5',
    'GS10','AAA','BAA','TB3SMFFM','TB6SMFFM','T1YFFM','T5YFFM','T10YFFM',
    'AAAFFM','BAAFFM','EXSZUSx','EXJPUSx','EXUSUKx','EXCAUSx',
    'PPICMM','UMCSENTx'
]
HORIZONS = [1, 5, 20]

print('Setup complete.')

In [ ]:
# ============================================================
# CELL 2: Data Loading & Prediction Helpers
# ============================================================

def load_macro_monthly(path, features):
    df = pd.read_csv(path, skiprows=[1]).rename(columns={'sasdate': 'date'})
    df['date'] = pd.to_datetime(df['date'])
    df = df.set_index('date').sort_index()
    keep = [c for c in features if c in df.columns]
    return df[keep].copy()

def load_price(path):
    px = pd.read_csv(path, parse_dates=['Date'])
    return px.set_index('Date')['Price'].sort_index()

def make_target(price, horizon=1):
    price = price.astype(float)
    return (np.log(price.shift(-horizon)) - np.log(price)).rename('y')

def build_sequences(macro_file, price_file, features, lookback=30, horizon=1,
                    train_end='2014-12-31', val_end='2024-12-31'):
    macro_m = load_macro_monthly(macro_file, features)
    macro_m.index = macro_m.index + pd.offsets.MonthBegin(1)
    price = load_price(price_file)
    macro_d = macro_m.reindex(price.index, method='ffill')
    base = macro_d.join(price, how='inner').dropna()
    base['ret_1d'] = np.log(base['Price']).diff()
    base = base.dropna()
    feat_cols = [c for c in features if c in base.columns] + ['ret_1d']
    feature_data = base[feat_cols].values
    target_data = make_target(base['Price'], horizon=horizon).values
    dates = base.index
    X_list, y_list, d_list = [], [], []
    for i in range(lookback, len(feature_data)):
        if i < len(target_data) and np.isfinite(target_data[i]):
            X_list.append(feature_data[i - lookback:i])
            y_list.append(target_data[i])
            d_list.append(dates[i])
    X_all = np.array(X_list, dtype=np.float32)
    y_all = np.array(y_list, dtype=np.float32)
    d_all = pd.DatetimeIndex(d_list)
    tr = d_all <= pd.Timestamp(train_end)
    va = (d_all > pd.Timestamp(train_end)) & (d_all <= pd.Timestamp(val_end))
    te = d_all > pd.Timestamp(val_end)
    return (X_all[tr], y_all[tr], X_all[va], y_all[va], X_all[te], y_all[te],
            d_all[tr], d_all[va], d_all[te], feat_cols)

class LSTMNet(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size,
                            num_layers=num_layers,
                            dropout=dropout if num_layers > 1 else 0.0,
                            batch_first=True)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden_size, 1))
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :]).squeeze(-1)

def load_model_and_predict(pkl_path, X_data, device='cpu'):
    with open(pkl_path, 'rb') as f:
        state = pickle.load(f)
    scaler = state['scaler_']
    cfg = state['model_data']['config']
    model = LSTMNet(cfg['input_size'], cfg['hidden_size'],
                    cfg['num_layers'], cfg['dropout']).to(device)
    model.load_state_dict(state['model_data']['state_dict'])
    model.eval()
    n, T, F = X_data.shape
    X_s = scaler.transform(X_data.reshape(-1, F)).reshape(n, T, F).astype(np.float32)
    with torch.no_grad():
        preds = model(torch.tensor(X_s).to(device)).cpu().numpy()
    return preds, state

def rmse(y, yhat): return float(np.sqrt(np.mean((y - yhat)**2)))
def mae(y, yhat): return float(np.mean(np.abs(y - yhat)))
def r2(y, yhat):
    ss_tot = np.sum((y - y.mean())**2)
    ss_res = np.sum((y - yhat)**2)
    return float(1 - ss_res / ss_tot) if ss_tot > 0 else np.nan
def directional_accuracy(y, yhat):
    return float(np.mean(np.sign(y) == np.sign(yhat)))

print('Helpers loaded.')

In [ ]:
# ============================================================
# CELL 3: Load All Data & Predictions
# ============================================================

data = {}  # {horizon: {Xtr, ytr, Xva, yva, Xte, yte, dva, dte, pred_val, pred_test, state}}

for h in HORIZONS:
    pkl_path = f'lstm_wti_h{h}.pkl'
    if not Path(pkl_path).exists():
        print(f'[SKIP] {pkl_path} not found')
        continue

    (Xtr, ytr, Xva, yva, Xte, yte,
     dtr, dva, dte, feat_cols) = build_sequences(
        MACRO_FILE, PRICE_FILE, FEATURES,
        lookback=30, horizon=h,
        train_end='2014-12-31', val_end='2024-12-31',
    )

    pred_val, state = load_model_and_predict(pkl_path, Xva)
    pred_test, _ = load_model_and_predict(pkl_path, Xte)

    data[h] = {
        'Xtr': Xtr, 'ytr': ytr,
        'Xva': Xva, 'yva': yva,
        'Xte': Xte, 'yte': yte,
        'dtr': dtr, 'dva': dva, 'dte': dte,
        'pred_val': pred_val, 'pred_test': pred_test,
        'state': state, 'feat_cols': feat_cols,
    }

    print(f'h={h:2d}d  |  Val: {len(yva)} samples  |  Test: {len(yte)} samples  |  Loaded ✓')

print(f'\nAll {len(data)} horizons loaded.')

In [ ]:
# ============================================================
# CELL 4: Predicted vs Actual — Time Series
# ============================================================

for h in HORIZONS:
    if h not in data: continue
    d = data[h]

    fig, axes = plt.subplots(2, 1, figsize=(15, 9), sharex=False)

    for ax, dates, y_true, y_pred, label in [
        (axes[0], d['dva'], d['yva'], d['pred_val'], 'Validation'),
        (axes[1], d['dte'], d['yte'], d['pred_test'], 'Test'),
    ]:
        ax.plot(dates, y_true, color=COLORS['actual'], linewidth=0.8, alpha=0.85, label='Actual')
        ax.plot(dates, y_pred, color=COLORS['pred'], linewidth=0.8, alpha=0.75, label='Predicted')
        ax.fill_between(dates, y_true, y_pred, alpha=0.10, color=COLORS['fill'])
        ax.axhline(0, color='gray', linewidth=0.5)
        ax.set_title(f'h={h}d — Predicted vs Actual ({label})', fontweight='bold')
        ax.set_ylabel(f'{h}-Day Log Return')
        ax.legend(loc='upper right', framealpha=0.9)

    axes[1].set_xlabel('Date')
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# CELL 5: Training & Validation Loss Curves
# ============================================================

def retrain_with_loss_tracking(X_train, y_train, best_params,
                               epochs=200, patience=20, batch_size=128,
                               device='cpu'):
    n = len(X_train)
    split = int(n * 0.9)
    Xtr, Xva = X_train[:split], X_train[split:]
    ytr, yva = y_train[:split], y_train[split:]

    # Scale
    _, T, F = Xtr.shape
    scaler = StandardScaler()
    scaler.fit(Xtr.reshape(-1, F))
    Xtr_s = scaler.transform(Xtr.reshape(-1, F)).reshape(len(Xtr), T, F).astype(np.float32)
    Xva_s = scaler.transform(Xva.reshape(-1, F)).reshape(len(Xva), T, F).astype(np.float32)

    model = LSTMNet(F, best_params['hidden_size'],
                    best_params['num_layers'], best_params['dropout']).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=best_params['lr'], weight_decay=1e-5)
    criterion = nn.MSELoss()

    train_ds = TensorDataset(torch.tensor(Xtr_s), torch.tensor(ytr))
    val_ds = TensorDataset(torch.tensor(Xva_s), torch.tensor(yva))
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=False)
    val_dl = DataLoader(val_ds, batch_size=batch_size * 2, shuffle=False)

    train_losses, val_losses = [], []
    best_val, wait = np.inf, 0

    for epoch in range(1, epochs + 1):
        model.train()
        ep_loss = 0.0
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            ep_loss += loss.item() * len(xb)
        train_losses.append(np.sqrt(ep_loss / len(train_ds)))

        model.eval()
        vp = []
        with torch.no_grad():
            for xb, _ in val_dl:
                vp.append(model(xb.to(device)).cpu().numpy())
        vp = np.concatenate(vp)
        vl = float(np.sqrt(np.mean((yva - vp)**2)))
        val_losses.append(vl)

        if vl < best_val:
            best_val = vl
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    return train_losses, val_losses

# --- Generate loss curves for each horizon ---
fig, axes = plt.subplots(1, len(HORIZONS), figsize=(6 * len(HORIZONS), 5))
if len(HORIZONS) == 1: axes = [axes]

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

for ax, h in zip(axes, HORIZONS):
    if h not in data: continue
    d = data[h]
    bp = d['state']['best_params_']
    print(f'Retraining h={h}d for loss curves...')
    train_l, val_l = retrain_with_loss_tracking(d['Xtr'], d['ytr'], bp, device=device)

    epochs = range(1, len(train_l) + 1)
    ax.plot(epochs, train_l, color=COLORS['h1'], linewidth=1.5, label='Train RMSE')
    ax.plot(epochs, val_l, color=COLORS['neg'], linewidth=1.5, label='Val RMSE')

    best_ep = np.argmin(val_l) + 1
    best_v = min(val_l)
    ax.axvline(best_ep, color='gray', linestyle=':', alpha=0.7)
    ax.annotate(f'Best: ep {best_ep}\n{best_v:.5f}',
                xy=(best_ep, best_v), fontsize=9,
                xytext=(best_ep + len(train_l)*0.08, best_v * 1.03),
                arrowprops=dict(arrowstyle='->', color='gray'),
                bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

    ax.set_title(f'h={h}d — Loss Curve', fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('RMSE')
    ax.legend(framealpha=0.9)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 6: Scatter — Predicted vs Actual
# ============================================================

for h in HORIZONS:
    if h not in data: continue
    d = data[h]

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    for ax, y_true, y_pred, label in [
        (axes[0], d['yva'], d['pred_val'], 'Validation'),
        (axes[1], d['yte'], d['pred_test'], 'Test'),
    ]:
        ax.scatter(y_true, y_pred, alpha=0.3, s=12, color=COLORS['h1'], edgecolors='none')

        # Limits
        lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
        margin = (lims[1] - lims[0]) * 0.05
        lims = [lims[0] - margin, lims[1] + margin]

        # 45-degree line
        ax.plot(lims, lims, '--', color='gray', linewidth=1, alpha=0.7, label='Perfect')

        # Regression line
        z = np.polyfit(y_true, y_pred, 1)
        x_line = np.linspace(lims[0], lims[1], 100)
        ax.plot(x_line, np.poly1d(z)(x_line), color=COLORS['pred'], linewidth=1.5,
                label=f'Fit: y={z[0]:.3f}x + {z[1]:.5f}')

        corr = np.corrcoef(y_true, y_pred)[0, 1]
        ax.text(0.05, 0.95, f'Corr: {corr:.4f}', transform=ax.transAxes,
                fontsize=10, va='top',
                bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

        ax.set_xlim(lims)
        ax.set_ylim(lims)
        ax.set_aspect('equal')
        ax.set_title(f'h={h}d — Scatter ({label})', fontweight='bold')
        ax.set_xlabel('Actual Return')
        ax.set_ylabel('Predicted Return')
        ax.legend(loc='lower right', fontsize=9, framealpha=0.9)

    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# CELL 7: Residual Analysis — Histogram + Q-Q Plot
# ============================================================

for h in HORIZONS:
    if h not in data: continue
    d = data[h]

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    for row, (y_true, y_pred, label) in enumerate([
        (d['yva'], d['pred_val'], 'Validation'),
        (d['yte'], d['pred_test'], 'Test'),
    ]):
        residuals = y_true - y_pred

        # Histogram
        ax = axes[row, 0]
        ax.hist(residuals, bins=50, color=COLORS['h5'], alpha=0.7,
                edgecolor='white', linewidth=0.3)
        ax.axvline(0, color=COLORS['neg'], linewidth=1.5, linestyle='--')
        ax.axvline(residuals.mean(), color=COLORS['h1'], linewidth=1.5,
                   label=f'Mean: {residuals.mean():.5f}')
        ax.set_title(f'h={h}d — Residuals ({label})', fontweight='bold')
        ax.set_xlabel('Residual (Actual − Predicted)')
        ax.set_ylabel('Count')
        ax.legend(framealpha=0.9)

        # Q-Q Plot
        ax2 = axes[row, 1]
        sorted_res = np.sort(residuals)
        n = len(sorted_res)
        theoretical = stats.norm.ppf(np.linspace(0.001, 0.999, n))
        ax2.scatter(theoretical, sorted_res, alpha=0.4, s=8,
                    color=COLORS['h1'], edgecolors='none')
        q25, q75 = np.percentile(sorted_res, [25, 75])
        t25, t75 = stats.norm.ppf(0.25), stats.norm.ppf(0.75)
        slope = (q75 - q25) / (t75 - t25)
        intercept = q25 - slope * t25
        ref_x = np.array([theoretical.min(), theoretical.max()])
        ax2.plot(ref_x, slope * ref_x + intercept, '--',
                 color=COLORS['neg'], linewidth=1.5)
        ax2.set_title(f'h={h}d — Q-Q Plot ({label})', fontweight='bold')
        ax2.set_xlabel('Theoretical Quantiles (Normal)')
        ax2.set_ylabel('Sample Quantiles')

    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# CELL 8: Cumulative Returns — Buy & Hold vs LSTM Strategy
# ============================================================

for h in HORIZONS:
    if h not in data: continue
    d = data[h]

    fig, axes = plt.subplots(2, 1, figsize=(15, 9))

    for ax, dates, y_true, y_pred, label in [
        (axes[0], d['dva'], d['yva'], d['pred_val'], 'Validation'),
        (axes[1], d['dte'], d['yte'], d['pred_test'], 'Test'),
    ]:
        cum_actual = np.cumsum(y_true)
        cum_long = np.cumsum(np.where(y_pred > 0, y_true, 0))
        cum_ls = np.cumsum(np.where(y_pred > 0, y_true, -y_true))

        ax.plot(dates, cum_actual, color=COLORS['actual'], linewidth=1.5,
                label='Buy & Hold')
        ax.plot(dates, cum_long, color=COLORS['pos'], linewidth=1.5,
                label='LSTM Long-Only')
        ax.plot(dates, cum_ls, color=COLORS['pred'], linewidth=1.5, alpha=0.7,
                label='LSTM Long/Short')
        ax.axhline(0, color='gray', linewidth=0.5)
        ax.fill_between(dates, cum_actual, alpha=0.05, color=COLORS['actual'])

        ax.set_title(f'h={h}d — Cumulative Returns ({label})', fontweight='bold')
        ax.set_ylabel('Cumulative Log Return')
        ax.legend(loc='best', framealpha=0.9)

    axes[1].set_xlabel('Date')
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# CELL 9: Rolling Directional Accuracy
# ============================================================

for h in HORIZONS:
    if h not in data: continue
    d = data[h]

    fig, axes = plt.subplots(2, 1, figsize=(15, 9))

    for ax, dates, y_true, y_pred, label, window in [
        (axes[0], d['dva'], d['yva'], d['pred_val'], 'Validation', 60),
        (axes[1], d['dte'], d['yte'], d['pred_test'], 'Test', min(30, len(d['yte'])//3)),
    ]:
        if len(y_true) < window:
            ax.text(0.5, 0.5, 'Not enough data', transform=ax.transAxes,
                    ha='center', fontsize=12)
            continue

        correct = (np.sign(y_true) == np.sign(y_pred)).astype(float)
        rolling = pd.Series(correct, index=dates).rolling(window, min_periods=window//2).mean()

        ax.plot(dates, rolling, color=COLORS['h1'], linewidth=1.2)
        ax.axhline(0.5, color=COLORS['neg'], linewidth=1.5, linestyle='--', label='50% (Random)')
        ax.fill_between(dates, rolling, 0.5, where=rolling > 0.5,
                        alpha=0.2, color=COLORS['pos'], label='Above 50%')
        ax.fill_between(dates, rolling, 0.5, where=rolling <= 0.5,
                        alpha=0.2, color=COLORS['neg'], label='Below 50%')
        ax.set_ylim(0.25, 0.75)
        ax.set_title(f'h={h}d — Rolling Dir. Accuracy, {window}-day ({label})', fontweight='bold')
        ax.set_ylabel('Directional Accuracy')
        ax.legend(loc='upper right', framealpha=0.9)

    axes[1].set_xlabel('Date')
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# CELL 10: CV Heatmaps
# ============================================================

fig, axes = plt.subplots(1, len(HORIZONS), figsize=(7 * len(HORIZONS), 6))
if len(HORIZONS) == 1: axes = [axes]

for ax, h in zip(axes, HORIZONS):
    if h not in data: continue
    cv_results = data[h]['state']['cv_results_']
    df = pd.DataFrame(cv_results)

    df['row'] = 'hs=' + df['hidden_size'].astype(str) + ' nl=' + df['num_layers'].astype(str)
    df['col'] = 'do=' + df['dropout'].astype(str) + ' lr=' + df['lr'].apply(lambda x: f'{x:.0e}')
    pivot = df.pivot(index='row', columns='col', values='cv_rmse')

    im = ax.imshow(pivot.values, cmap='YlOrRd', aspect='auto')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=9)

    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            val = pivot.values[i, j]
            color = 'white' if val > pivot.values.mean() else 'black'
            ax.text(j, i, f'{val:.4f}', ha='center', va='center',
                    fontsize=7, color=color, fontweight='bold')

    best_idx = np.unravel_index(pivot.values.argmin(), pivot.values.shape)
    ax.add_patch(plt.Rectangle((best_idx[1]-0.5, best_idx[0]-0.5), 1, 1,
                                fill=False, edgecolor=COLORS['pos'], linewidth=3))

    plt.colorbar(im, ax=ax, label='CV RMSE', shrink=0.8)
    ax.set_title(f'h={h}d — CV Heatmap', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 11: Multi-Horizon Comparison Bar Chart
# ============================================================

metrics_list = ['rmse', 'mae', 'r2', 'directional_acc']
titles = ['RMSE ↓', 'MAE ↓', 'R² ↑', 'Directional Accuracy ↑']

# Compute metrics
all_m = {}
for h in HORIZONS:
    if h not in data: continue
    d = data[h]
    all_m[h] = {
        'val':  {m: f(d['yva'], d['pred_val']) for m, f in
                 zip(metrics_list, [rmse, mae, r2, directional_accuracy])},
        'test': {m: f(d['yte'], d['pred_test']) for m, f in
                 zip(metrics_list, [rmse, mae, r2, directional_accuracy])},
    }

horizons = sorted(all_m.keys())
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
x = np.arange(len(horizons))
width = 0.35

for i, (metric, title) in enumerate(zip(metrics_list, titles)):
    ax = axes[i]
    val_vals  = [all_m[h]['val'][metric] for h in horizons]
    test_vals = [all_m[h]['test'][metric] for h in horizons]

    bars1 = ax.bar(x - width/2, val_vals, width, label='Validation',
                   color=COLORS['bar1'], alpha=0.85)
    bars2 = ax.bar(x + width/2, test_vals, width, label='Test',
                   color=COLORS['bar2'], alpha=0.85)

    ax.set_title(title, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([f'h={h}d' for h in horizons])

    if metric == 'directional_acc':
        ax.axhline(0.5, color='gray', linestyle='--', linewidth=1, alpha=0.7)
    elif metric == 'r2':
        ax.axhline(0, color='gray', linestyle='--', linewidth=1, alpha=0.7)

    for bar in list(bars1) + list(bars2):
        ht = bar.get_height()
        ax.annotate(f'{ht:.3f}', xy=(bar.get_x() + bar.get_width()/2, ht),
                   xytext=(0, 3), textcoords='offset points',
                   ha='center', va='bottom', fontsize=8)

    if i == 0:
        ax.legend(framealpha=0.9)

plt.suptitle('LSTM Multi-Horizon Comparison', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 12: Summary Metrics Table
# ============================================================

rows = []
for h in HORIZONS:
    if h not in all_m: continue
    for split in ['val', 'test']:
        m = all_m[h][split]
        rows.append({
            'Horizon': f'{h}d',
            'Split': split.title(),
            'RMSE': f"{m['rmse']:.6f}",
            'MAE': f"{m['mae']:.6f}",
            'R²': f"{m['r2']:.6f}",
            'Dir. Acc': f"{m['directional_acc']:.4f}",
        })

summary = pd.DataFrame(rows)
print('\n' + '='*65)
print('  LSTM Multi-Horizon Results Summary')
print('='*65)
print(summary.to_string(index=False))
print('='*65)

# Best configs
print('\nBest Hyperparameters:')
for h in HORIZONS:
    if h not in data: continue
    bp = data[h]['state']['best_params_']
    print(f"  h={h:>2}d: hs={bp['hidden_size']}, nl={bp['num_layers']}, "
          f"do={bp['dropout']}, lr={bp['lr']}, cv_rmse={bp['cv_rmse']:.6f}")